# 00b — Phân tích kết quả G0b (không cần GPU)

Đọc `results.jsonl` của run G0b và in lại mọi con số mà sổ dự án dùng (§4 áp quy tắc G0, §9 kết quả G0b). Mục đích: con số trong sổ có nguồn tái tạo được; ai cũng chạy lại được trên máy (chỉ cần numpy, pandas).

Phân tích thêm so với notebook 00 (sổ D-11, D-12):
- **Hệ số co biên độ c:** hồi quy logit diff từng câu của Q theo FP (m_Q = c·m_FP + b), như cách đo *margin shrinkage* của Wu et al. (arXiv 2608.06564). So T với c và với tỉ lệ logit diff trung bình.
- **T theo từng ô** ở vùng 1, 2, 5, để xem chỗ nào hỏng trước.
- **Lưới nén** theo quy tắc G0 số 2.

In [ ]:
# parameters
RESULTS = "results/20261113_0506_g0b/g0b_full_410m/results.jsonl"
LATTICE = [[4, 1.0], [3, 1.0], [2, 0.5], [2, 0.75]]   # lưới đề xuất, kiểm theo quy tắc 2
OUT_DIR = "outputs/00b_local"

## 1. Đọc kết quả

In [ ]:
import json, os
import numpy as np
import pandas as pd

os.makedirs(OUT_DIR, exist_ok=True)
if not os.path.exists(RESULTS) and os.path.exists(os.path.join("..", RESULTS)):
    RESULTS = os.path.join("..", RESULTS)   # mở notebook từ thư mục notebooks/
rows = [json.loads(l) for l in open(RESULTS, encoding="utf-8")]
by = lambda e: [r for r in rows if r["exp"] == e]
meta, dc = by("meta")[0], by("data_check")[0]
print("model", meta["model"], "| task", meta["task"], "| commit notebook bản", meta["notebook_version"],
      "| n_eval", meta["n_eval"], "| cụm", meta["n_eval_clusters"])
print("data_check:", {k: dc[k] for k in ("first_token_is_eos", "tok_mismatch", "eval_sentences_in_train", "eval_swap_frac")})
print("ví dụ:", dc["example_spans"], "| nhãn", dc["example_labels"])

## 2. Bản đồ odds FP (DAS, seed 0)

Hàng: layer; cột: vùng (1 chủ ngữ, 2 ` that`, 3 động từ mệnh đề, 4 ` the`, 5 tân ngữ = vị trí dự đoán). Chỉ số kiểu bài gốc: max theo vùng ở mỗi layer, trung bình qua layer.

In [ ]:
fp = {(r["layer"], r["region"]): r["odds"] for r in by("das_fp")}
regions = sorted({k[1] for k in fp}); layers = sorted({k[0] for k in fp})
grid = pd.DataFrame([[fp.get((l, r), np.nan) for r in regions] for l in layers], index=layers,
                    columns=[f"r{r}" for r in regions])
print(grid.round(2).to_string())
print("\nodds kiểu bài gốc:", round(by("fp_map_summary")[0]["odds_maxpos_meanlayer"], 2),
      "| bài gốc (Table 11, 410m, tác vụ này): 4.97")
ms = by("main_site")[0]
print("ô chính theo D-08:", (ms["layer"], ms["region"]), "| odds", round(ms["odds"], 2), "|", ms["rule"])

## 3. Hành vi theo cấu hình nén và hệ số co biên độ c

- `ratio_ld`: logit diff trung bình của Q chia cho FP.
- `c`: độ dốc hồi quy logit diff từng câu của Q theo FP; `b` là chặn. Nếu nén chỉ làm co biên độ đầu ra, odds khi can thiệp cũng co theo c (b triệt tiêu khi lấy hiệu), nên T ≈ c dù không gian con không đổi.
- `c_lo`, `c_hi`: khoảng tin cậy 95% của c, bootstrap theo cụm (mỗi cụm 2 câu của một cặp).
- `R_lo`, `R_hi`: khoảng tin cậy 95% của `ratio_ld` (ký hiệu R), cùng cách bootstrap.
- `T_main_over_R`, `T_r5_over_R`: T chia R. Đề xuất D-12 dùng R làm mẫu số chính vì R hẹp hơn nhiều so với c (c ước từ độ dốc nên nhiễu khi logit diff của FP ít phân tán).

In [ ]:
beh = {(r["bits"], r["alpha"]): r for r in by("behavior")}
ld_fp = np.asarray(beh[(None, 0.0)]["ld"])
ncl = len(ld_fp) // 2
# câu thứ i thuộc cụm i // 2 theo thứ tự ghi của notebook 00 (base rồi src của từng cặp gốc)
clusters = np.arange(len(ld_fp)) // 2
rng = np.random.default_rng(0)

def slope(x, y):
    return float(np.polyfit(x, y, 1)[0])

def boot_ratio(x, y, n=2000):
    groups = [np.flatnonzero(clusters == c) for c in np.unique(clusters)]
    vals = []
    for _ in range(n):
        p = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        vals.append(y[p].mean() / x[p].mean())
    return np.percentile(vals, [2.5, 97.5])

def boot_slope(x, y, n=2000):
    groups = [np.flatnonzero(clusters == c) for c in np.unique(clusters)]
    vals = []
    for _ in range(n):
        p = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        vals.append(slope(x[p], y[p]))
    return np.percentile(vals, [2.5, 97.5])

reuse = {(r["bits"], r["alpha"]): r for r in by("reuse_map")}
tab = []
for k, r in reuse.items():
    q = np.asarray(beh[k]["ld"])
    c = slope(ld_fp, q); lo, hi = boot_slope(ld_fp, q); R = q.mean() / ld_fp.mean(); R_lo, R_hi = boot_ratio(ld_fp, q)
    m = {(a, b): (o, u) for a, b, o, u in r["map"]}
    t5 = [m[(l, 5)][1] / m[(l, 5)][0] for l in layers if (l, 5) in m and m[(l, 5)][0] >= 1]
    tab.append(dict(bits=k[0], alpha=k[1], w_rel_err=r["w_rel_err"], acc=beh[k]["acc"], logit_diff=beh[k]["logit_diff"],
                    ratio_ld=R, R_lo=R_lo, R_hi=R_hi, c=c, c_lo=lo, c_hi=hi, b=float(np.polyfit(ld_fp, q, 1)[1]),
                    T_main=r["transfer_T"], T_lo=r["transfer_T_lo"], T_hi=r["transfer_T_hi"],
                    T_cells_median=r["t_cells_median"], T_r5_median=float(np.median(t5)),
                    T_main_over_R=r["transfer_T"] / R, T_r5_over_R=float(np.median(t5)) / R,
                    T_main_over_c=r["transfer_T"] / c))
tab = pd.DataFrame(tab)
pd.set_option("display.width", 250)
print(tab.round(3).to_string(index=False))
print("\nTương quan qua", len(tab), "cấu hình:")
print("  corr(ratio_ld, T_cells_median) =", round(np.corrcoef(tab.ratio_ld, tab.T_cells_median)[0, 1], 4))
print("  corr(ratio_ld, T_main)         =", round(np.corrcoef(tab.ratio_ld, tab.T_main)[0, 1], 4))
print("  corr(c, T_r5_median)           =", round(np.corrcoef(tab.c, tab.T_r5_median)[0, 1], 4))
print("FP: 2 x logit diff TB =", round(2 * ld_fp.mean(), 2), "(trần của một phép tráo trung thực; odds lớn hơn số này là DAS vượt trần)")
tab.to_csv(os.path.join(OUT_DIR, "g0b_margin.csv"), index=False)

## 4. T theo từng ô ở vùng 1, 2, 5

T = odds khi dùng lại hướng FP trên Q chia odds FP, ở các ô có odds FP ≥ 1. Một seed DAS cho mỗi ô, chưa có khoảng tin cậy theo ô [GỢI Ý].

In [ ]:
for reg in (1, 2, 5):
    out = []
    for k, r in reuse.items():
        m = {(a, b): (o, u) for a, b, o, u in r["map"]}
        out.append([f"{k[0]}b a{k[1]:.2f}"] + [round(m[(l, reg)][1] / m[(l, reg)][0], 2)
                   if (l, reg) in m and m[(l, reg)][0] >= 1 else None for l in layers])
    df = pd.DataFrame(out, columns=["cfg"] + [f"L{l}" for l in layers]).set_index("cfg").dropna(axis=1, how="all")
    print(f"\nVùng {reg}:\n", df.to_string())

## 5. Học lại trên Q, dao động seed, sàn step0, tách đoạn

Tất cả ở ô chính D-08 (L22, vùng 5). Ô này sát đầu ra; tách đoạn ở đây suy biến (phía sau chỉ còn 1 khối), xem sổ D-11.

In [ ]:
ss = by("seed_spread")[0]
print("Seed FP (k=%d): odds %.2f-%.2f; |cos| min %.3f, TB %.3f" % (ss["k"], ss["odds_min"], ss["odds_max"], ss["abs_cos_min"], ss["abs_cos_mean"]))
fl = by("floor_step0")[0]
print("Sàn step0: odds %.3f = %.1f%% odds FP" % (fl["odds"], 100 * fl["ratio_to_fp"]))
rl = pd.DataFrame([dict(bits=r["bits"], alpha=r["alpha"], odds_mean=r["odds_mean"], ratio_to_fp=r["ratio_to_fp"],
                        cos_within_fp=r["abs_cos_within_fp_mean"], cos_within_q=r["abs_cos_within_q_mean"],
                        cos_cross=r["abs_cos_cross_mean"], cos_cross_max=max(r["abs_cos_cross"]),
                        cos_within_fp_min=min(ss["abs_cos"])) for r in by("relearn_q")])
print("\nHọc lại trên Q (k = 5 seed):\n", rl.round(3).to_string(index=False))
sp = pd.DataFrame([dict(part=r["part"], bits=r["bits"], acc=r["acc"], T=r["transfer_T"], T_lo=r["transfer_T_lo"],
                        T_hi=r["transfer_T_hi"]) for r in by("split_q")])
print("\nTách đoạn ở L22 (pre = layer 0-22, post = layer 23):\n", sp.round(3).to_string(index=False))
tm = by("timing")[0]
print("\nThời gian: học %.2f s + đánh giá %.2f s = %.2f s/ô; 3.000 lần = %.1f giờ; 10.000 lần = %.1f giờ"
      % (tm["t_train_median_s"], tm["t_eval_median_s"], tm["t_per_run_s"], tm["gpu_hours_3000_runs"], tm["gpu_hours_10000_runs"]))

## 6. Lưới nén theo quy tắc G0 số 2

Yêu cầu: 3–4 mức, accuracy giảm dần từ ≥ 0,95 × FP xuống gần ngẫu nhiên (accuracy ≤ 0,55 hoặc logit diff ≤ 0,1 × FP). Ưu tiên mức bit thật (α = 1); chỉ dùng α khi không có mức bit nào rơi vào khoảng cần.

In [ ]:
acc_fp, ld0 = beh[(None, 0.0)]["acc"], ld_fp.mean()
sel = [beh[(int(b), float(a))] for b, a in LATTICE]
for r in sel:
    print(f"{r['bits']} bit, alpha {r['alpha']:.2f}: acc {r['acc']:.2f} ({r['acc'] / acc_fp:.2f} x FP), logit diff {r['logit_diff']:.3f}, w_rel_err {r['w_rel_err']:.3f}")
accs = [r["acc"] for r in sel]
ok = (accs[0] >= 0.95 * acc_fp and all(a > b for a, b in zip(accs, accs[1:]))
      and (accs[-1] <= 0.55 or sel[-1]["logit_diff"] <= 0.1 * ld0) and 3 <= len(sel) <= 4)
print("Quy tắc 2:", "đạt" if ok else "không đạt")
b2 = beh.get((2, 1.0))
if b2:
    print(f"Ghi chú: 2 bit alpha 1.0 có acc {b2['acc']:.2f} > alpha 0.75 ({beh[(2, 0.75)]['acc']:.2f}): không đơn điệu, không dùng làm mức.")